In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import ast

## Data cleaning

Exploration of datasets to verify the accuracy of the data, to perform necessary conversions, to analyze duplicate information and NaN values and their management, with possible elimination of empty columns and replacement of NaNs with medians or relevant data to the analysis.

### Character anime works

In [2]:
character_anime_works = pd.read_csv(r'../data/character_anime_works.csv')
character_anime_works.tail()

,anime_mal_id,character_mal_id,character_name,role
236811,31245,137157,"Shibasaki, Ken",Supporting
236812,36305,136064,"Hamanaka, Midori",Main
236813,36305,133916,"Narumi, Sena",Main
236814,36305,124942,"Hayasaka, Akari",Supporting
236815,34895,155887,"Shiraha, Setsuna",Main


This dataset represents a bridge table between Anime and Characters, especially the "who plays what" relationship and defines the nature of that participation through the role column.

In [3]:
character_anime_works.shape

(236816, 4)

In [4]:
character_anime_works.columns

Index(['anime_mal_id', 'character_mal_id', 'character_name', 'role'], dtype='object')

In [5]:
character_anime_works.dtypes

anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

In [6]:
character_anime_works['role'].value_counts()

role
Supporting    186285
Main           50531
Name: count, dtype: int64

Observations:
- **Relational structure**: essential to let us answer to qestions about the highest number of main character of an anime and many more.
- **Role distribution**: natural assignment of supporting and main roles in a narrative dataset.

To do:
- **Memory usage**: if anime_mal_id and character_mal_id have values representable using int32 we can save memory space. Furthermore, 'role' column has two standard values, it'd be more efficient if we could convert it into 'category' type.
- **Duplicates management**: an essential requirement includes if the same character is associated to the same anime more than once. This could be a repeated information error.
- **NaN analysis**: verify potentially ID missing in some rows to avoid existence of dirty data.

In [7]:
# Memory usage
# Checking max values (Scaling up strategy)
print(f"Max anime_id: {character_anime_works['anime_mal_id'].max()}")
print(f"Max character_id: {character_anime_works['character_mal_id'].max()}")

# Saving memory space
character_anime_works['anime_mal_id'] = character_anime_works['anime_mal_id'].astype('int32')
character_anime_works['character_mal_id'] = character_anime_works['character_mal_id'].astype('int32')

Max anime_id: 62584
Max character_id: 282284


In [8]:
character_anime_works.dtypes

anime_mal_id         int32
character_mal_id     int32
character_name      object
role                object
dtype: object

In [9]:
# Type conversion for higher efficiency
character_anime_works['role'] = character_anime_works['role'].astype('category')

# Memory usage
character_anime_works.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236816 entries, 0 to 236815
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   anime_mal_id      236816 non-null  int32   
 1   character_mal_id  236816 non-null  int32   
 2   character_name    236816 non-null  object  
 3   role              236816 non-null  category
dtypes: category(1), int32(2), object(1)
memory usage: 3.8+ MB


In [10]:
# Duplicates management - key couple
duplicates = character_anime_works.duplicated(subset=['anime_mal_id', 'character_mal_id']).sum()
print(f"Relazioni duplicate: {duplicates}")

# Removal keeping the first occurency
character_anime_works = character_anime_works.loc[~character_anime_works.duplicated(subset=['anime_mal_id', 'character_mal_id'])]

Relazioni duplicate: 0


In [11]:
# NaN analysis
# Counting NaNs in key column
critical_nans = character_anime_works[['anime_mal_id', 'character_mal_id']].isna().sum()
print("NaN in colonne critiche:\n", critical_nans)

# Removing rows due to unimportant meaning
# character_anime_works = character_anime_works.dropna(subset=['anime_mal_id', 'character_mal_id'])

NaN in colonne critiche:
 anime_mal_id        0
character_mal_id    0
dtype: int64


Results:
- **Lighter**: Memory requirements will be reduced by approximately 60-70% thanks to the int32 and category types.

- **More robust**: We'll be sure that every line represents a valid relationship between an anime and a character.

- **User-centered**: We've removed "noise" (duplicates and potential null IDs) to allow journalists to draw accurate conclusions.

In [12]:
# Overwriting/importing clean dataframe into .csv
character_anime_works.to_csv('../data/character_anime_works_clean.csv', index=False)
print(f"\nSaved clean characher_anime_works dataset")


Saved clean characher_anime_works dataset


In [48]:
character_anime_works = None
print(f"\nRemoving charachers_anime_works dataset from RAM")


Removing charachers_anime_works dataset from RAM


Implementation ideas:
1. Counting how much anime each character act in
2. Which anime has the most crowded cast; a journalist may desire to know if anime with more characters in has higher or lower ratings cross-checking data with details.csv
3. Creating a filtered view with only main characters

### Characters' details

In [23]:
characters = pd.read_csv(r'../data/characters.csv')
characters

,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN
...,...,...,...,...,...,...,...
209958,282276.0,https://myanimelist.net/character/282276/Farra...,Farrah Van Dorothy,ファラ・ヴァン・ドロシー,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209959,282277.0,https://myanimelist.net/character/282277/Harri...,Harris Mead,ハリス・ミード,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209960,282278.0,https://myanimelist.net/character/282278/Rob,Rob,ロブ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209961,282281.0,https://myanimelist.net/character/282281/Grimm,Grimm,グリム,https://cdn.myanimelist.net/images/characters/...,0.0,NaN


In [24]:
characters.shape

(209963, 7)

In [25]:
characters.columns

Index(['character_mal_id', 'url', 'name', 'name_kanji', 'image', 'favorites',
       'about'],
      dtype='object')

In [26]:
characters.dtypes

character_mal_id    float64
url                  object
name                 object
name_kanji           object
image                object
favorites           float64
about                object
dtype: object

Duplicated ID - repeated informations error.

Based on our notes during the course, in this case we should remove them keeping only the first occurrence to avoid changing mean and other statistics

In [27]:
# Counts duplicated ID
duplicate_count = characters['character_mal_id'].duplicated().sum()
print(f"ID duplicati trovati: {duplicate_count}")

# To see duplicated rows
if duplicate_count > 0:
    print(characters[characters['character_mal_id'].duplicated(keep=False)].sort_values(by='character_mal_id'))

ID duplicati trovati: 456
        character_mal_id                                                url  \
208187             546.0       https://myanimelist.net/character/546/Hayate   
208185             546.0       https://myanimelist.net/character/546/Hayate   
208186             546.0       https://myanimelist.net/character/546/Hayate   
204358            4527.0  https://myanimelist.net/character/4527/Narue_N...   
204356            4527.0  https://myanimelist.net/character/4527/Narue_N...   
...                  ...                                                ...   
209850          282115.0    https://myanimelist.net/character/282115/Cavitt   
209859          282131.0  https://myanimelist.net/character/282131/Lieut...   
209858          282131.0  https://myanimelist.net/character/282131/Lieut...   
209733               NaN                                                NaN   
209734               NaN                                                NaN   

                     name

In [28]:
# Removing duplicates and keeping first record found
characters = characters.loc[~characters.duplicated(subset=['character_mal_id'])]

# Confirm current status
print(f"Dataset dopo la rimozione duplicati: {len(characters)} righe")

Dataset dopo la rimozione duplicati: 209507 righe


Column 'favorites' - meaning: describing how many times a user add the character to his/her/its favourites

In [29]:
characters['favorites'].describe()

count    209506.000000
mean         57.697216
std        1198.841827
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max      175632.000000
Name: favorites, dtype: float64

1. **High percentage of zero values**: usual event of large-scale dataset: a  few characters (the famous protagonists) have thousands of likes, while the vast majority of secondary characters have zero or very few of them.
2. **Type anomaly**: The dtype is float64, but the "likes" are necessary integers.

Solutions
- **Memory optimization**: convert column values into int32. Float64 value type takes up a lot of space.

- **Outlier management**: the maximum value of 175.632 is an important signal (the most loved character), not an error. Don't remove but use it to generate a categorization (es. "Top Tier", "Mainstream", "Niche") to encourage analysis for expert users.

- **Cardinality Filter**: If you're only interested in relevant people for your "Data Story", you could create a view of the data frame that excludes those with zero favorites, reducing the cardinality; this allows you to "work with seemingly smaller data" and focus only on the portion of data actually needed for popularity analysis.

In [36]:
# Memory optimization - replacing NaN values with 0 (not in favorites) and convert
characters.loc[:, 'favorites'] = characters['favorites'].fillna(0).astype('int32')
print(characters['favorites'].dtypes)

float64


In [37]:
# Outlier management - identify outlier categories
bins = [0, 1000, 10000, float('inf')]
label = ['Niche', 'Mainstream', 'Top Tier']
characters.loc[:, 'popularity'] = pd.cut(characters['favorites'], bins=bins, labels=label)
print(characters['popularity'].value_counts())

popularity
Niche         76268
Mainstream     1326
Top Tier        231
Name: count, dtype: int64


In [38]:
# Cardinality Filter - generating a view which avoid rows with 'favorites' values = 0
characters_fav_view = characters[characters['favorites'] > 0]

# If we don't need original data, reallocate to save memory
# characters = characters[characters['favorites'] > 0]

print(f"View rows (character with fav > 0): {len(characters_fav_view)}")

View rows (character with fav > 0): 77825


This convert the highest value to essential knowledge to the audience.

Column 'about' meaning - character's biography

In [39]:
characters['about'].value_counts()

about
No voice actors have been added to this character. Help improve our database by searching for a voice actor, and adding this character to their roles .                                                                                                                                                                                                                                                                                                                                                                                                                            199
Appears in episode 1.                                                                                                                                                                                                                                                                                                                                                                                                                      

**Data Interpretation**: the 199 occurrences of "No voice actors have been added..." and short descriptions like "Appears in episode 1" are signs of poor or missing data, even though they are not formally NaNs.

Thoughts
1. **Replacement**: Is it better to replace Nan with "No info"?
2. **Length Analysis**: Instead of reading the text, create a new column about_length. There's often a correlation between a character's popularity and the length of their description.
3. **Radical Memory Savings**: If you don't plan on doing text analysis, this column takes up a lot of RAM. Out notes suggest that if it's not needed for the current analysis, it's best to drop it or not load it at all to save space.

In [40]:
# Replacement - improves readability
characters.loc[:, 'about'] = characters['about'].fillna('No info')

If a journalist reads a profile, he/she'll be albe to understand the information more clearly than an empty space or a technical error.

In [42]:
# Total for each column
print(characters.isna().sum())

# Total of each row which contains at least a NaN value
print(f"Totale righe con almeno un NaN: {characters.isna().any(axis=1).sum()}")

character_mal_id         1
url                      1
name                     1
name_kanji           55369
image                    1
favorites                0
about                    0
popularity          131682
dtype: int64
Totale righe con almeno un NaN: 143990


The data extracted shows that 125,436 of 209,963 rows (about 60% of the dataset) have at least one null value.

What does this mean?

- **Massive Loss**: If you used a simple dropna(), you would delete more than half of your database.

- **Nature of the data**: Most of the NaNs are found in _about_ (96,976) and _name_kanji_ (55,480). It's common for many characters to lack a biography or Japanese name in the database.

To do:
- **Avoid Bias**: If we remove a row due to a NaN in an irrelevant column (such as name_kanji if you're doing a statistical analysis of favorites), out analysis will be highly biased, i.e., incorrect.

- **Leverage Pandas' design**: Pandas is designed to skip NaNs during numeric operations, so their presence will not affect the calculation of the favorites average.

- **Critical columns**: Only remove rows if the NaN is present in character_mal_id or name, since without this data the element becomes meaningless.

In [43]:
# We only remove if essential identity data is missing
characters = characters.dropna(subset=['character_mal_id'])

# For the others, we do not delete anything to avoid bias
print(f"Percentuale di dati preservati: {len(characters)/209963*100:.2f}%")

Percentuale di dati preservati: 99.78%


In [44]:
# Overwriting/importing clean dataframe into .csv
characters.to_csv('../data/characters_clean.csv', index=False)
print(f"\nSaved clean charachers dataset")


Saved clean charachers dataset


In [47]:
characters = None
print(f"\nRemoving charachers dataset from RAM")


Removing charachers dataset from RAM


### Details

In [47]:
details_df = pd.read_csv(r'../data/Details.csv')
details_df.tail()

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
28950,59421,Zutaboro Reijou wa Ane no Moto Konyakusha ni D...,ずたぼろ令嬢は姉の元婚約者に溺愛される,https://myanimelist.net/anime/59421/Zutaboro_R...,https://cdn.myanimelist.net/images/anime/1518/...,TV,Finished Airing,7.37,15624.0,2025-07-05T00:00:00+00:00,...,['Josei'],Light novel,PG-13 - Teens 13 or older,12.0,summer,2025.0,"['Studio Pierrot', 'Mainichi Broadcasting Syst...",[],[],"['Crunchyroll', 'Aniplus TV', 'Bahamut Anime C..."
28951,31245,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～,https://myanimelist.net/anime/31245/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/3/821...,Movie,Finished Airing,7.20,104106.0,2016-04-23T00:00:00+00:00,...,[],Music,PG-13 - Teens 13 or older,1.0,NaN,NaN,"['Aniplex', 'Dentsu', 'Kadokawa Shoten', 'Movi...",[],['Aniplex of America'],[]
28952,36305,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～ 「金曜日のおはよう」,https://myanimelist.net/anime/36305/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/6/883...,Special,Finished Airing,7.17,10038.0,2016-10-26T00:00:00+00:00,...,[],Music,PG - Children,1.0,NaN,NaN,['Aniplex'],[],[],[]
28953,34895,Zutto Suki Datta,ずっと好きだった,https://myanimelist.net/anime/34895/Zutto_Suki...,https://cdn.myanimelist.net/images/anime/1498/...,OVA,Finished Airing,5.68,1887.0,2017-04-21T00:00:00+00:00,...,[],Manga,Rx - Hentai,2.0,NaN,NaN,"['Queen Bee', 'Mediabank']",[],[],[]
28954,56606,Zutto Tomodachi,ずっと友達,https://myanimelist.net/anime/56606/Zutto_Tomo...,https://cdn.myanimelist.net/images/anime/1462/...,Music,Finished Airing,NaN,NaN,1996-06-01T00:00:00+00:00,...,['Kids'],Original,G - All Ages,1.0,NaN,NaN,['NHK'],[],[],[]


### GESTIONE ANNO DI RILASCIO, DUPLICATI E CONVERSIONE DATE
In questa fase preliminare verifichiamo l'unicità dei dati attraverso l'identificativo univoco mal_id. Sono presenti 2 coppie di duplicati attraverso gli'identificativi 'title' e 'type', andando a verificare hanno 'mal_id' e 'start-date' differenti quindi non li consideriamo duplicati.
Successivamente, convertiamo le colonne temporali (start_date, end_date) nel formato corretto datetime. Sovrascriviamo la colonna originale year, utilizzando l'anno da start_date, poichè presentava un alto numero di valori mancanti. Utilizziamo errors='coerce' per trasformare eventuali date anomale in NaT (Not a Time), garantendo la coerenza del dataset.


In [48]:
initial_rows = len(details_df)
details_df.drop_duplicates(subset='mal_id', keep='first', inplace=True)
print(f"Duplicati rimossi: {initial_rows - len(details_df)}")

details_df['start_date'] = pd.to_datetime(details_df['start_date'], errors='coerce')
details_df['end_date'] = pd.to_datetime(details_df['end_date'], errors='coerce')
details_df['year'] = details_df['start_date'].dt.year
details_df['year'] = details_df['year'].astype('Int64')

subset_check = ['title', 'type']
duplicates_mask = details_df.duplicated(subset=subset_check, keep=False)

num_duplicates = duplicates_mask.sum()
print(f"Possibili duplicati trovati (stesso Titolo e Tipo): {num_duplicates}")
if num_duplicates > 0:
    print("\nEsempio di righe duplicate (Titolo + Tipo):")
    # Ordiniamo per titolo così vediamo le coppie vicine
    display(details_df[duplicates_mask].sort_values(by='title')[['mal_id', 'title', 'type', 'start_date']].head(10))

Duplicati rimossi: 0
Possibili duplicati trovati (stesso Titolo e Tipo): 4

Esempio di righe duplicate (Titolo + Tipo):


,mal_id,title,type,start_date
16205,59994,Moonlight,Music,2024-09-30 00:00:00+00:00
16206,60524,Moonlight,Music,2024-12-09 00:00:00+00:00
21601,53367,Shen Lan Qi Yu Wushuang Zhu: Tianmo Pian,ONA,2023-01-14 00:00:00+00:00
21602,55740,Shen Lan Qi Yu Wushuang Zhu: Tianmo Pian,ONA,2023-07-15 00:00:00+00:00


In [53]:
print(details_df.dtypes)

mal_id                           int64
title                           object
type                            object
status                          object
score                          float64
scored_by                      float64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                           float64
popularity                       int64
members                          int64
favorites                        int64
genres                          object
studios                         object
themes                          object
demographics                    object
source                          object
rating                          object
episodes                       float64
year                             Int64
producers                       object
explicit_genres                 object
licensors                       object
streaming                       object
dtype: object


### CONVERSIONE STRINGHE E GESIONE VALORI MANCANTI
- In questa fase abbiamo trasformato le colonne come 'genres' e 'studios' utilizzando la libreria ast, convertendole in vere e proprie liste Python, permettendo al software di riconoscere ogni singolo genere o studio separatamente. Questa operazione è fondamentale per contare correttamente quante volte compare un genere o confrontare i vari studi di produzione tra loro.
- Nelle colonne convertite in liste, come i generi, i valori nulli sono stati sostituiti con liste vuote per mantenere la coerenza del codice e permettere calcoli sicuri. Per quanto riguarda le valutazioni numeriche come lo score, abbiamo scelto di conservare i valori mancanti invece di sostituirli con degli zeri, evitando così di abbassare artificialmente la media globale dei voti e preservando l'integrità della distribuzione.

In [50]:
# Funzione di conversione sicura
def to_list(x):
    if isinstance(x, str) and x.startswith('['):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return x if isinstance(x, list) else []

details_df['genres'] = details_df['genres'].apply(to_list)
details_df['studios'] = details_df['studios'].apply(to_list)
details_df['producers'] = details_df['producers'].apply(to_list)
details_df['licensors'] = details_df['licensors'].apply(to_list)
details_df['themes'] = details_df['themes'].apply(to_list)
details_df['demographics'] = details_df['demographics'].apply(to_list)

# Verifica del tipo di dato per una colonna
print(f"Tipo di dato in 'genres': {type(details_df['genres'].iloc[0])}")

Tipo di dato in 'genres': <class 'list'>


### RIMOZIONE RIGHE E COLONNE INUTILI
In quest'ultima fase abbiamo rimosso i record privi di informazioni essenziali come il titolo e il tipo di produzione, poiché la loro assenza impedirebbe qualsiasi classificazione corretta dell'opera. Abbiamo inoltre eliminato le colonne superflue (title_japanese, image_url, url, season) per ridurre l'occupazione di memoria e focalizzare l'analisi sulle variabili chiave. La verifica finale mostra un dataset solido di 28.888 righe, dove le lacune residue (come negli score) sono fisiologiche e verranno gestite puntualmente durante le analisi statistiche per non inquinare i risultati.

In [52]:
details_df.dropna(subset=['title', 'type'], inplace=True)

colonne_da_escludere = ['title_japanese','image_url', 'url', 'season']
details_df.drop(columns=colonne_da_escludere, axis=1, errors='ignore', inplace=True)

# 5. VERIFICA FINALE
null_counts = details_df.isna().sum()
print("Dataset pulito con successo!")
print(f"Righe rimaste: {len(details_df)}")
print("\nColonne con ancora valori mancanti (%):")
print((null_counts[null_counts > 0] / len(details_df)) * 100)

Dataset pulito con successo!
Righe rimaste: 28888

Colonne con ancora valori mancanti (%):
score         34.640681
scored_by     34.640681
start_date     2.748546
end_date      61.001108
synopsis      17.723622
rank          23.861119
rating         1.896981
episodes       2.132373
year           2.748546
dtype: float64


In [55]:
output_name = 'Details_Cleaned.csv'
details_df.to_csv(output_name, index=False)
print(f"\nDataset pulito salvato come: {output_name}")


Dataset pulito salvato come: Details_Cleaned.csv
